In [1]:
import json
import pandas as pd
import numpy as np
import glob
import os

def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    json_files = glob.glob(os.path.join(input_folder, "*.json"))
    print(f" Found {len(json_files)} JSON files.")

    for json_file in json_files:
        base = os.path.splitext(os.path.basename(json_file))[0]
        out_csv = os.path.join(output_folder, f"{base}.csv")

        print(f"\nProcessing: {json_file}")

        gps_samples, ypr_samples = [], []

        # === Load JSON ===
        with open(json_file, "r") as f:
            data = json.load(f)

        # === Parse metadata ===
        for stream_id, stream_data in data.items():
            if isinstance(stream_data, dict) and "streams" in stream_data:
                streams = stream_data["streams"]

                # ================= GPS ====================
                gps_stream = streams.get("GPS5")
                if gps_stream and "samples" in gps_stream:
                    for s in gps_stream["samples"]:
                        val = s.get("value",[])

                        if not isinstance(val,list):
                            continue

                        
                        if len(val) >= 3:
                            gps_samples.append({
                                "file": base,
                                "stream_id": stream_id,
                                "cts": s.get("cts"),
                                "datetime": s.get("date"),
                                "lat": val[0],
                                "lon": val[1],
                                "alt": val[2] if len(val) > 2 else None,
                                "spd_3d": val[3] if len(val) > 3 else None,
                                "spd_2d": val[4] if len(val) > 4 else None
                            })

                # ================= YPR ====================
                for key in ["CORI", "GYRO", "ISOG", "CAMERA_ORIENTATION"]:
                    if key in streams:
                        ypr_stream = streams[key]
                        if ypr_stream and "samples" in ypr_stream:
                            for s in ypr_stream["samples"]:
                                val = s.get("value", [])
                                if not isinstance(val,list):
                                    continue
                                if len(val) >= 3:
                                    ypr_samples.append({
                                        "file": base,
                                        "stream_id": stream_id,
                                        "cts": s.get("cts"),
                                        "datetime": s.get("date"),
                                        "yaw": val[0],
                                        "pitch": val[1],
                                        "roll": val[2]
                                    })

        # === DataFrame ===
        gps_df = pd.DataFrame(gps_samples)
        ypr_df = pd.DataFrame(ypr_samples)

        if gps_df.empty:
            print(f"No GPS data in {json_file}. Skipping.")
            continue

        gps_df["datetime"] = pd.to_datetime(gps_df["datetime"], errors="coerce")
        if not ypr_df.empty:
            ypr_df["datetime"] = pd.to_datetime(ypr_df["datetime"], errors="coerce")

        # === Relative time ===
        start_time = gps_df["datetime"].min()
        gps_df["t_rel_sec"] = (gps_df["datetime"] - start_time).dt.total_seconds()

        if not ypr_df.empty:
            ypr_df["t_rel_sec"] = (ypr_df["datetime"] - start_time).dt.total_seconds()

            # === Interpolate YPR ===
            for col in ["yaw", "pitch", "roll"]:
                gps_df[col] = np.interp(
                    gps_df["t_rel_sec"],
                    ypr_df["t_rel_sec"],
                    ypr_df[col]
                )
        else:
            gps_df["yaw"] = np.nan
            gps_df["pitch"] = np.nan
            gps_df["roll"] = np.nan

        # === Placeholder sensor fields ===
        for col in ["gyro_x", "gyro_y", "gyro_z", "accel_x", "accel_y", "accel_z", "temp"]:
            gps_df[col] = np.nan

        # === Save ===
        gps_df.to_csv(out_csv, index=False)
        print(f" Saved {len(gps_df)} rows → {out_csv}")

    print("\n All processing completed!")

process_folder(r"E:/04.Pak_ival/Dataset/merdeka_2/1.360_raw/metadata", r"E:/04.Pak_ival/Dataset/merdeka_2/1.360_raw/output")


 Found 1 JSON files.

Processing: E:/04.Pak_ival/Dataset/merdeka_2/1.360_raw/metadata\GS014429.json
 Saved 9026 rows → E:/04.Pak_ival/Dataset/merdeka_2/1.360_raw/output\GS014429.csv

 All processing completed!
